In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import re
from collections import Counter
from datasets import load_dataset
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# -------------------------
# 1. Load Full NLP Dataset
# -------------------------
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
text = " ".join(dataset["train"]["text"])
text = text.lower()
text = re.sub(r'[^a-zA-Z ]', ' ', text)
text = re.sub(r'\s+', ' ', text)
words = text.split()

# -------------------------
# 2. Build Larger Vocabulary
# -------------------------
word_counts = Counter(words)
most_common = word_counts.most_common(30000)  # top 30k words
vocab = [w for w,_ in most_common]
word_to_idx = {w:i for i,w in enumerate(vocab)}
word_to_idx["<unk>"] = len(vocab)
idx_to_word = {i:w for w,i in word_to_idx.items()}
vocab_size = len(word_to_idx)

# Map words to indices
words_idx = [word_to_idx.get(w, word_to_idx["<unk>"]) for w in words]

# -------------------------
# 3. Create Sequences
# -------------------------
sequence_length = 5
inputs = []
targets = []

for i in range(len(words_idx) - sequence_length):
    seq = words_idx[i:i+sequence_length]
    target = words_idx[i+sequence_length]
    inputs.append(seq)
    targets.append(target)

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)

# -------------------------
# 4. DataLoader
# -------------------------
batch_size = 128
train_dataset = TensorDataset(inputs, targets)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print(f"Training samples: {len(inputs)}, Vocabulary size: {vocab_size}")

# -------------------------
# 5. LSTM Model
# -------------------------
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

model = LSTMModel(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

# -------------------------
# 6. Train the Model
# -------------------------
epochs = 20

for epoch in range(epochs):
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg_loss = total_loss / len(train_dataset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# -------------------------
# 7. Generate Text with Sampling
# -------------------------
def generate_text(seed, length=50, top_k=10):
    model.eval()
    words_gen = seed.lower().split()
    for _ in range(length):
        seq = [word_to_idx.get(w, word_to_idx["<unk>"]) for w in words_gen[-sequence_length:]]
        seq = torch.tensor(seq).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(seq)
            probs = torch.softmax(logits, dim=-1)
            # Top-k sampling
            top_probs, top_idx = torch.topk(probs, top_k)
            top_probs = top_probs.squeeze(0)
            top_idx = top_idx.squeeze(0)
            next_word_idx = top_idx[torch.multinomial(top_probs, 1).item()].item()
        words_gen.append(idx_to_word[next_word_idx])
    return " ".join(words_gen)

# -------------------------
# 8. Test Generation
# -------------------------
print("\nGenerated Sequence:\n")
print(generate_text("machine learning models"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Training samples: 1694389, Vocabulary size: 30001
